# SIH26165 — GLiNER + SetFit fine-tuning on Colab (T4 / L4)

Runtime → Change runtime type → **GPU**.  
Two ways to bring the data in: mount Google Drive (recommended, checkpoints survive disconnects) or upload a zip.

What to upload from the laptop (after `filter_distilled` + `prepare_*` have run):

```
DATA/distilled/gliner/{train,dev,test}.json + labels.json   (+ weak_train.json if made)
DATA/distilled/setfit/{train,dev,test}.jsonl
DATA/Processed/gold/gold.jsonl, DATA/Processed/ihm/ihm_eval.jsonl
TRAINING/  (the whole package)
SERVER/Classfication/Models/RAW/gliner_multi/   (optional — otherwise urchade/gliner_multi-v2.1 is pulled from the hub)
```
Zip it on the laptop: `python -m TRAINING.pack_for_colab` → `sih_train_bundle.zip` (see README).

In [ ]:
#@title 1. Install (≈2 min)
!pip -q install "transformers>=4.40,<5" "sentence-transformers>=3.0" "setfit>=1.1" "gliner>=0.2.13" "datasets>=2.18" accelerate scikit-learn joblib sentencepiece protobuf
import torch; print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")

In [ ]:
#@title 2. Get the bundle  (pick ONE of the two blocks)
USE_DRIVE = True  #@param {type:"boolean"}
BUNDLE_ON_DRIVE = "/content/drive/MyDrive/SIH_26/sih_train_bundle.zip"  #@param {type:"string"}
import os, shutil
if USE_DRIVE:
    from google.colab import drive; drive.mount("/content/drive")
    src = BUNDLE_ON_DRIVE
else:
    from google.colab import files; up = files.upload(); src = next(iter(up))
os.makedirs("/content/SIH_26", exist_ok=True)
shutil.unpack_archive(src, "/content/SIH_26")
%cd /content/SIH_26
!ls && ls DATA/distilled/gliner DATA/distilled/setfit

In [ ]:
#@title 3. (Drive only) keep checkpoints on Drive so a disconnect does not lose the epoch
if USE_DRIVE:
    ck = "/content/drive/MyDrive/SIH_26/checkpoints"; os.makedirs(ck, exist_ok=True)
    if os.path.islink("TRAINING/checkpoints") or not os.path.exists("TRAINING/checkpoints"):
        !rm -rf TRAINING/checkpoints && ln -s $ck TRAINING/checkpoints
    !ls -la TRAINING/checkpoints

In [ ]:
#@title 4. Smoke test (1 min) - proves the package + data load
!python -m TRAINING.finetune.train_gliner --smoke --device cuda

In [ ]:
#@title 5. GLiNER fine-tune  (T4: ~25-40 min for 8k rows x 3 epochs; resume-safe)
EPOCHS = 3      #@param {type:"integer"}
BATCH  = 8      #@param {type:"integer"}
LR     = 1e-5   #@param {type:"number"}
!python -m TRAINING.finetune.train_gliner --epochs $EPOCHS --batch $BATCH --lr $LR --device cuda --resume

In [ ]:
#@title 6. GLiNER eval on test split (span P/R/F1 per role)
!python -m TRAINING.finetune.train_gliner --eval-only --device cuda
!cat SERVER/Classfication/Models/Tunned/gliner_sif/dev_metrics.json | head -60

In [ ]:
#@title 7. SetFit heads on GPU (optional - the CPU 'head' mode is the default on the laptop)
TASKS = "prefilter,verdict,statement_type,lsr"  #@param {type:"string"}
!python -m TRAINING.finetune.train_setfit --mode setfit --tasks $TASKS --epochs 1 --iterations 20 --batch 32 --device cuda

In [ ]:
#@title 8. Gold-180 + IHM evaluation
!python -m TRAINING.eval.evaluate_gold
!cat TRAINING/eval/reports/gold_eval.md

In [ ]:
#@title 9. Pack the trained models for the laptop -> SERVER/Classfication/Models/Tunned
!zip -qr sif_models.zip SERVER/Classfication/Models/Tunned TRAINING/eval/reports
if USE_DRIVE:
    shutil.copy("sif_models.zip", "/content/drive/MyDrive/SIH_26/sif_models.zip"); print("saved to Drive: SIH_26/sif_models.zip")
else:
    from google.colab import files; files.download("sif_models.zip")

Back on the laptop: unzip `sif_models.zip` at the repo root (`E:\SIH_26`) — it drops `gliner_sif/` and `sif_heads/` into `SERVER/Classfication/Models/Tunned/`.